In [30]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np

In [31]:
with open('persuasion.txt') as f:
    content = f.read()
print(len(content))

484132


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [33]:
allowed_chars = set(string.ascii_letters + string.digits + ". ")

content = ''.join([c for c in content if c in allowed_chars])

chars = set(sorted(list(content)))
i_to_c = {k:v for k, v in enumerate(chars)}
c_to_i = {k:v for v, k in i_to_c.items()}

encode = lambda s: [c_to_i[c] for c in s] # noqa: E731
decode = lambda s: ''.join([i_to_c[i] for i in s]) # noqa: E731
decode(encode(content[:100]))

encoded_content = torch.tensor(encode(content))

In [34]:
blocks = []
block_length = 6
batch_size = 32

xs = []
ys = []

for i in range(0, len(encoded_content)-block_length):
    x_chunk = encoded_content[i:i+block_length-1]
    y_chunk = encoded_content[i+1:i+block_length]

    xs.append(torch.tensor(x_chunk))
    ys.append(torch.tensor(y_chunk))

X = F.one_hot(torch.stack(xs)).float()
Y = torch.stack(ys)


/tmp/ipykernel_32343/3824314043.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xs.append(torch.tensor(x_chunk))
/tmp/ipykernel_32343/3824314043.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ys.append(torch.tensor(y_chunk))


In [35]:
X = X.to(device)
Y = Y.to(device)

In [40]:

# x_input @ W +b1
hidden_layer_width = 512
relu = torch.nn.ReLU()

W1 = torch.randn(len(chars), hidden_layer_width, device=device)*0.01
W1.requires_grad_(True)

b1 = torch.zeros(hidden_layer_width, requires_grad=True, device=device)

W2 = torch.randn(hidden_layer_width, len(chars), device=device)*0.01
W2.requires_grad_(True)

b2 = torch.zeros(len(chars), requires_grad=True, device=device)


In [37]:
device

device(type='cuda')

In [42]:


import time

timings = []
for epoch in range(1):

    for i in range(0, len(X), batch_size):
        torch.cuda.synchronize()
        start = time.perf_counter()
        indices = torch.randperm(len(X))
        indices = indices.to(device)
        batch_idx = indices[i:i+batch_size]
        

        x = X[batch_idx]
        y = Y[batch_idx]
        z1 = x @ W1 + b1
        h1 = relu(z1)
        logits = h1 @ W2 + b2
        loss = F.cross_entropy(logits.view(-1, len(chars)), y.view(-1))


        loss.backward()
        
        if W1.grad is not None:
            for p in [W1, W2, b1, b2]: 
                p.grad.zero_()


        with torch.no_grad():
            for p in [W1, W2, b1, b2]:
                p -= 0.01 * p.grad
        
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        if not i%1000:
            print(f'epoch {epoch}, loss {loss}, {i} of {len(X)} iterations')
            print(f'time for a single iter: {elapsed:.6f}')
            timings.append(elapsed)
            print(f'average time per iter so far: {np.mean(timings)}')

epoch 0, loss 4.158673286437988, 0 of 463240 iterations
time for a single iter: 0.500827
average time per iter so far: 0.5008271449996755
epoch 0, loss 4.158631324768066, 4000 of 463240 iterations
time for a single iter: 0.010711
average time per iter so far: 0.25576918999968257
epoch 0, loss 4.158481597900391, 8000 of 463240 iterations
time for a single iter: 0.012864
average time per iter so far: 0.17480068033304028
epoch 0, loss 4.158812522888184, 12000 of 463240 iterations
time for a single iter: 0.012328
average time per iter so far: 0.13418257924968202
epoch 0, loss 4.158537864685059, 16000 of 463240 iterations
time for a single iter: 0.014780
average time per iter so far: 0.11030199959968741
epoch 0, loss 4.1586785316467285, 20000 of 463240 iterations
time for a single iter: 0.015726
average time per iter so far: 0.09453928816628832
epoch 0, loss 4.158792018890381, 24000 of 463240 iterations
time for a single iter: 0.011215
average time per iter so far: 0.08263584828533307
epoch

KeyboardInterrupt: 

In [ ]:
inputs = 'h'

for i in range(100):
    x = F.one_hot(torch.tensor(encode(inputs[-1])), num_classes=len(chars))

    z1 = x.float() @ W1 + b1
    relu = torch.nn.ReLU()
    h = relu(z1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=-1)
    idx = torch.multinomial(probs, num_samples=1).item()
    inputs+=i_to_c[idx]
    # output = decode(logits.argmax(dim=1).tolist())

    # inputs+=output
    print(inputs)



he
he 
he t
he th
he the
he thed
he thed 
he thed f
he thed fe
he thed fe 
he thed fe k
he thed fe k 
he thed fe k h
he thed fe k hl
he thed fe k hl 
he thed fe k hl i
he thed fe k hl il
he thed fe k hl ild
he thed fe k hl ild 
he thed fe k hl ild t
he thed fe k hl ild tt
he thed fe k hl ild tt 
he thed fe k hl ild tt .
he thed fe k hl ild tt . 
he thed fe k hl ild tt . R
he thed fe k hl ild tt . R 
he thed fe k hl ild tt . R t
he thed fe k hl ild tt . R th
he thed fe k hl ild tt . R th 
he thed fe k hl ild tt . R th u
he thed fe k hl ild tt . R th ug
he thed fe k hl ild tt . R th ug 
he thed fe k hl ild tt . R th ug B
he thed fe k hl ild tt . R th ug Bo
he thed fe k hl ild tt . R th ug Bow
he thed fe k hl ild tt . R th ug Bowu
he thed fe k hl ild tt . R th ug Bowu 
he thed fe k hl ild tt . R th ug Bowu l
he thed fe k hl ild tt . R th ug Bowu le
he thed fe k hl ild tt . R th ug Bowu le 
he thed fe k hl ild tt . R th ug Bowu le b
he thed fe k hl ild tt . R th ug Bowu le ba
he thed fe k 